<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day15_practice3_%EA%B0%90%EC%84%B1%EB%B6%84%EB%A5%98_NSMC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 네이버 영화 리뷰 감성 분류

In [2]:
from torch._C import device
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import urllib.request, os
from collections import Counter # Counter=단어 빈도 세기

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# 셀 1. NSMC - 네이버 영화 리뷰
URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
urllib.request.urlretrieve(URL, "ratings_train.txt")
print("ratings_train.txt 다운로드 완료")
texts, labels = [], []
with open("ratings_train.txt", encoding = "utf-8") as f:
  next(f) # 첫 줄(헤더 id document label)을 한 번 읽어 버림
  for line in f:
    parts = line.strip().split("\t")
    if len(parts) == 3 and parts[1]:
      texts.append(parts[1])
      labels.append(int(parts[2]))
print(f"전체 리뷰 {len(texts):,}개 | 긍정 {sum(labels):,}/부정 {len(labels)-sum(labels):,}")
print("샘플:", texts[0], "→", "긍정" if labels[0] else "부정")

N = 30000
texts, labels = texts[:N], labels[:N] # 3만개만 슬라이싱

ratings_train.txt 다운로드 완료
전체 리뷰 149,995개 | 긍정 74,825/부정 75,170
샘플: 아 더빙.. 진짜 짜증나네요 목소리 → 부정


In [11]:
# 셀 2. vocab - '빈도 상위'만
counter = Counter(tok for t in texts for tok in t.split()) # {단어: 등장횟수}
print(f"고유 어절 수: {len(counter):,}개")
print("최다 빈도:", counter.most_common(5))

VOCAB_SIZE = 15000
vocab = {"<pad>": 0, "<unk>": 1}
for tok, _ in counter.most_common(VOCAB_SIZE - 2):
  vocab[tok] = len(vocab) # 번호 부여

MAX_LEN = 20
def encode(text):
  ids = [vocab.get(t, 1) for t in text.split()][:MAX_LEN] # 단어를 번호로 바꾸고 너무 길면 자른다
  return ids + [0] * (MAX_LEN - len(ids)) # 너무 짧으면 뒤를 0으로 채운다
print(f"\n원문: {texts[0]!r}")
print(f"[변수 확인] encode(texts[0]) = {encode(texts[0])}") # 첫 리뷰 문장의 단어를 번호로 바꾼거

X = torch.tensor([encode(t) for t in texts])
y = torch.tensor(labels, dtype = torch.float32).reshape(-1,1)

print(f"\nX.shape = {X.shape}")
print(f"y.shape = {y.shape}")

print(f"X[0] = {X[0]}")
print(f"y[0] = {y[0]}")

# 학습/평가 분리
n_train = int(N * 0.9)
train_loader = DataLoader(TensorDataset(X[:n_train], y[:n_train]), batch_size = 256, shuffle=True) # 앞 90% # [256, 20] :문장, 단어
X_test, y_test = X[n_train:].to(device), y[n_train:].to(device) # 뒤 10%

print(f"X_test.shape = {X_test.shape}, y_test.shape = {y_test.shape}")

고유 어절 수: 98,463개
최다 빈도: [('영화', 2241), ('너무', 1602), ('정말', 1566), ('진짜', 1191), ('이', 1021)]

원문: '아 더빙.. 진짜 짜증나네요 목소리'
[변수 확인] encode(texts[0]) = [51, 1, 5, 10248, 1557, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

X.shape = torch.Size([30000, 20])
y.shape = torch.Size([30000, 1])
X[0] = tensor([   51,     1,     5, 10248,  1557,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0])
y[0] = tensor([0.])
X_test.shape = torch.Size([3000, 20]), y_test.shape = torch.Size([3000, 1])


In [15]:
# 셀 3. 모델 - 임베딩 + 평균 + MLP
class SentimentNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.emb = nn.Embedding(len(vocab), 64, padding_idx=0) # 임베딩 표(단어수×64차원벡터)
    self.fc = nn.Sequential(
        nn.Linear(64, 32), nn.ReLU(), # 입력 64차원 벡터가 한 문장 안에 있는 단어들의 벡터를 평균으로 하나로 합친 것
        nn.Linear(32, 1), nn.Sigmoid() # 긍정/부정
    )
  def forward(self, x, debug_mode=False): # x: (B, 20) 각 문장이 단어번호 20개
    emb = self.emb(x)
    if debug_mode:
      print(f"emb shape: {emb.shape}")
      print(f"emb 전체:")
      print(emb)

    # 진짜 단어 3개 + pad 2개인데 5로 나누면 → 값이 희석돼 작아진다. '진짜 단어만' 골라 평균
    mask = (x != 0).unsqueeze(-1).float() # (B, 20, 1)   1. 마스크 만들기:어디가 진짜 단어이고 어디가 pad인가, True/False 표
    if debug_mode:
      print(f"\nmask shape: {mask.shape}")
      print(f"mask 전체:")
      print(mask)

    vecs = emb * mask # (B,20,64) 각 문장의 단어수 20×64차원벡터   2. 단어 벡터를 꺼내고 pad자리는 0으로 지우기
    if debug_mode:
      print(f"\nvecs shape: {vecs.shape}")
      print(f"vecs 전체:")
      print(vecs)

    sent = vecs.sum(1)/mask.sum(1).clamp(min=1) # (B,64) 벡터들의 합/진짜 단어 개수   3. '진짜 단어 개수'로만 나눠 평균, clamp(min=1)나누는 값이 최소1
    if debug_mode:
      print(f"\nsent shape: {sent.shape}")
      print(f"sent 전체:")
      print(sent)

    return self.fc(sent) # (B,64) 문장벡터(한 문장에 있는 단어들의 벡터들 하나로 다 합친것 평균을 내서) → 긍정확률(B,1)

model = SentimentNet().to(device)
loss_fn = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.002)
print(f"모델 파라미터: {sum(p.numel() for p in model.parameters())/1e3:.0f}K" f"(임베딩 표가 {len(vocab)*64/1e3:.0f}K)")

모델 파라미터: 962K(임베딩 표가 960K)


In [16]:
# 모델 디버그 모드 사용 예시
print("--- 디버그 모드 실행 결과 ---")
dummy_input = torch.tensor([[10, 20, 0, 0, 0], [5, 15, 25, 0, 0]]).to(device) # 더미 입력 생성, 각 열은 문장 내 단어의 위치
_ = model(dummy_input, debug_mode=True)
print("--- 버그 모드 실행 완료 ---")

--- 디버그 모드 실행 결과 ---
emb shape: torch.Size([2, 5, 64])
emb 전체:
tensor([[[-1.4229, -0.2340,  0.6629,  1.0754,  0.2657, -0.3312,  0.1052,
          -0.4793,  0.4483,  0.2760, -1.4507,  1.1450,  0.9817,  0.9667,
          -0.0556,  0.8373,  1.3883,  1.1851,  1.0805, -0.1019, -0.4525,
           0.4516,  1.4903,  0.6467, -0.9071, -2.1621, -1.2994, -0.3630,
          -1.5214,  0.9276,  0.5061,  0.3730,  1.5901,  0.9360,  1.1440,
          -0.5092,  0.0842,  0.2645, -0.5958,  0.5847,  0.4956,  0.9426,
          -0.7195, -0.4279,  1.1078, -1.9538,  0.2030,  1.3630,  1.0700,
           0.0552, -0.2049, -0.4337,  0.9661, -0.9830,  0.0251, -0.1599,
          -0.6256, -0.0909, -0.3909,  1.8400,  1.1177,  1.0000, -0.4743,
           1.1228],
         [ 1.0824, -0.8553, -0.1275,  2.5494, -0.2427,  0.6378,  0.1221,
           2.2305, -0.4002, -1.1243, -0.8278, -0.0872, -0.5214,  1.1607,
           1.2599,  0.7211, -1.2714,  1.0757, -0.1608,  0.0554, -1.1482,
           0.3861, -1.8927,  0.7368, -0.2

In [17]:
# 셀 4. 학습 + 평가
EPOCHS = 5
for epoch in range(EPOCHS):
  model.train()
  for xb, yb in train_loader: # 배치 사이즈만큼 주입
    xb, yb = xb.to(device), yb.to(device)
    loss = loss_fn(model(xb), yb) # 256 문장
    opt.zero_grad()
    loss.backward()
    opt.step()
  model.eval()
  with torch.no_grad():
    acc = ((model(X_test) > 0.5) == y_test.bool()).float().mean().item()
  print(f"epoch {epoch+1}/{EPOCHS} | 테스트 정확도 {acc:.4f}")

epoch 1/5 | 테스트 정확도 0.6143
epoch 2/5 | 테스트 정확도 0.6977
epoch 3/5 | 테스트 정확도 0.7287
epoch 4/5 | 테스트 정확도 0.7353
epoch 5/5 | 테스트 정확도 0.7413


In [18]:
# 셀 5. 새 문장 예측
def predict(sentence):
  model.eval()
  with torch.no_grad():
    p = model(torch.tensor([encode(sentence)]).to(device)).item()
  return f"{'긍정' if p > 0.5 else '부정'} ({p:.0%})"

for s in ["정말 재미있고 감동적인 영화", "시간이 아깝다 정말 지루함",
          "배우 연기가 최고", "스토리가 엉망이다"]:
          print(f"{s!r} → {predict(s)}")

'정말 재미있고 감동적인 영화' → 긍정 (100%)
'시간이 아깝다 정말 지루함' → 부정 (0%)
'배우 연기가 최고' → 긍정 (100%)
'스토리가 엉망이다' → 부정 (3%)


In [20]:
# 셀 6. 맹점 발견 - 순서가 사라졌다
# 우리 모델의 문장 벡터 = '단어 평균'
# 단어 구성이 같으면(순서만 달라도) 평균은 완전히 동일
a = "재미가 없지 않다"
b = "재미가 있지 않다"
print(f"{a!r} → {predict(a)}")
print(f"{b!r} → {predict(b)}")

s1 = "지루하다 하지만 결말은 최고"
s2 = "최고 하지만 결말은 지루하다"
print(f"{s1!r} → {predict(s1)}")
print(f"{s2!r} → {predict(s2)}")

'재미가 없지 않다' → 부정 (6%)
'재미가 있지 않다' → 부정 (4%)
'지루하다 하지만 결말은 최고' → 부정 (11%)
'최고 하지만 결말은 지루하다' → 부정 (11%)
